# M17 · Transformer basics

Curriculum · Domain 4 · LLMs

**Use attention to let each token decide which other tokens matter.**

In this notebook we implement scaled dot-product attention from scratch with NumPy. The core formula is

$$\text{softmax}(QK^\top/\sqrt{d})V$$

No model weights, downloads, or GPUs are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(17)

## Tiny token embeddings

We pretend three ads/query tokens already have 4-dimensional embeddings. A real transformer learns projections into queries, keys, and values; here we choose small deterministic matrices so the math is visible.

In [ ]:
tokens = ["search", "ads", "relevance"]
X = np.array([
    [1.0, 0.2, 0.0, 0.1],
    [0.9, 0.1, 0.3, 0.0],
    [0.0, 0.1, 1.0, 0.2],
])

W_q = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.0],
    [0.0, 0.5],
])
W_k = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.4, 0.0],
    [0.0, 0.4],
])
W_v = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0],
])

Q = X @ W_q
K = X @ W_k
V = X @ W_v

print(Q)

## Step 1 - Scores

For every token pair, compute $QK^\top$. A high score means the query vector and key vector point in similar directions.

In [ ]:
scores = Q @ K.T
scaled_scores = scores / np.sqrt(Q.shape[1])

print(np.round(scaled_scores, 3))

assert scaled_scores.shape == (3, 3)

## Step 2 - Row-wise softmax

Softmax turns each row into attention weights. Every row sums to 1, so each output vector is a weighted average of value vectors.

In [ ]:
def softmax_rows(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

A = softmax_rows(scaled_scores)

print(np.round(A, 3))

assert np.allclose(A.sum(axis=1), 1.0)

## Step 3 - Weighted values

Now multiply the attention matrix by $V$. Each output row is the context-aware version of the corresponding input token.

In [ ]:
O = A @ V

for token, vector in zip(tokens, O):
    print(token, np.round(vector, 3))

assert O.shape == (3, 2)

## Add a causal mask

A decoder cannot look into the future. We set future logits to a very negative number before softmax, which makes their weights effectively zero.

In [ ]:
mask = np.triu(np.ones_like(scaled_scores), k=1).astype(bool)
causal_scores = scaled_scores.copy()
causal_scores[mask] = -1e9
causal_A = softmax_rows(causal_scores)

print(np.round(causal_A, 3))

assert np.allclose(causal_A[mask], 0.0)

## Visualize the attention matrix

The heatmap shows which tokens each token listens to. Rows are listeners; columns are sources of information.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(A, vmin=0.0, vmax=1.0, cmap="Blues")
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_title("self-attention weights")
fig.colorbar(im, ax=ax)
plt.show()

## Practice

1. Change `W_q` so the token `relevance` pays more attention to `ads`.
2. Increase the sequence to four tokens and confirm the attention matrix becomes $4\times4$.
3. Compare encoder attention `A` with decoder attention `causal_A` for the final token.

In [ ]:
# Your turn:
